<a href="https://colab.research.google.com/github/leejsya/FashionMNIST_Classification/blob/main/Modeling_FashionMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Library & Module Import

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

### Transforms

In [ ]:
''' Preprocessing for train '''
transforms_train = v2.Compose(
    [
    # 이미지를 텐서로 변환
    v2.ToImage(),
    # 이미지 좌우반전 - 확률 기본값=0.5
    v2.RandomHorizontalFlip(),
    # 이미지 일부분 잘라내기 - 잘라낸 후 28크기로 resize
    v2.RandomResizedCrop(size=28, scale=(0.8, 1.0)),
    # 이미지 회전 - 최대 10도만큼
    v2.RandomRotation(degrees=10),
    # 픽셀값 데이터형 변환 및 스케일링 - float32, 0과 1 사이의 값으로 스케일링
    v2.ToDtype(dtype=torch.float32, scale=True),
    # Noramlize - 기존의 평균과 표준편차를 기준으로 각각 0과 1로 변환
    v2.Normalize(mean=[0.286], std=[0.353]),
    ]
)

''' Preprocessing for eval '''
transforms_eval = v2.Compose(
    [
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale=True),
    v2.Normalize(mean=[0.286], std=[0.353]),
    ]
)

mean, std 다시 확인

### Dataset

In [ ]:
''' Train Dataset '''
train_dataset = datasets.FashionMNIST(
    root='./', # 데이터셋이 저장될 위치
    train=True,     # train dataset 추출
    download=True,  # 데이터셋 다운로드하기
    transform=transforms_train  # transform 적용해서 데이터셋 가져오기
)

''' Validation Dataset '''
val_dataset = datasets.FashionMNIST(
    root='./',
    train=True,
    download=True,
    transform=transforms_eval
)

''' Test Dataset '''
test_dataset = datasets.FashionMNIST(
    root='./',
    train=False,   # test dataset 추출
    download=True,
    transform=transforms_eval
)

''' stratum 비율을 유지하도록 인덱스를 분할'''
train_indices, val_indices = train_test_split(
    range(len(train_dataset)),
    test_size=0.1, # 6 : 2 : 2
    stratify=train_dataset.targets,
    random_state=5
    # + shuffle
)

''' Subset으로 인덱스에 해당하는 부분들을 가리키도록 '''
train_dataset = Subset(train_dataset, train_indices) # +) mean, std 다시 구하기
val_dataset = Subset(val_dataset, val_indices)

100%|██████████| 26.4M/26.4M [00:01<00:00, 14.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 211kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.92MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.2MB/s]


### DataLoader

In [ ]:
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=128,
    shuffle=True,
    drop_last=True,
)
val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
)
test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
)

### Model Class(LesNet-5)

In [ ]:
class LeNetModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Conv2d(입력채널, 출력채널, 커널사이즈, Stride, Padding)
        self.conv1 = nn.Conv2d(1, 6, 5, 1, 2)
        self.conv2 = nn.Conv2d(6, 16, 5, 1, 0)

        # MaxPool2d(윈도우 사이즈, Stride)
        self.maxpool = nn.MaxPool2d(2, 2)

        # 텐서를 벡터로 정렬
        self.flatten = nn.Flatten()

        # FC Layers
        self.linear1 = nn.Linear(400, 120)
        self.linear2 = nn.Linear(120, 84)
        self.linear3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.maxpool(x)

        x = self.flatten(x)

        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)
        x = F.relu(x)
        output = self.linear3(x)

        return output

### Model Class(ResNet)

In [ ]:
# class Block(nn.Module):
#     def __init__(self, in_ch, out_ch, stride=1):
#         super().__init__()
#         self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
#         self.bn1 = nn.BatchNorm2d(out_ch)
#         self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
#         self.bn2 = nn.BatchNorm2d(out_ch)

#         self.shortcut = nn.Identity()
#         if stride != 1 or in_ch != out_ch:
#             self.shortcut = nn.Sequential(
#                 nn.Conv2d(in_ch, out_ch, 1 , stride, bias=False),
#                 nn.BatchNorm2d(out_ch),
#             )

#     def forward(self, x):
#         x = self.conv1(x)
#         x = self.bn1(x)
#         x = F.relu(x)
#         x = self.conv2(x)
#         x = self.bn2(x)
#         x = x + self.shortcut(x)
#         x = F.relu(x)

#         return x



# class ResNetModel(nn.Module):
#     def __init__(self, num_blocks=(2, 2, 2), num_classes=10):
#         super().__init__()
#         self.stem = nn.Sequential(
#             nn.Conv2d(1, 16, 3, 1, 1, bias=False),
#             nn.BatchNorm2d(16),
#             nn.ReLU(),
#         )

#         self.layer1 = self._make_layer(16, 16, num_blocks[0], stride=1)
#         self.layer2 = self._make_layer(16, 32, num_blocks[1], stride=2)
#         self.layer3 = self._make_layer(32, 64, num_blocks[2], stride=2)
#         self.pool = nn.AdaptiveAvgPool2d(1)
#         self.fc = nn.Linear(64, num_classes)

#     def _make_layer(self, in_ch, out_ch, n, stride):
#         layers = [Block(in_ch, out_ch, stride)]
#         layers += [Block(out_ch, out_ch, 1) for _ in range(n - 1)]
#         return nn.Sequential(*layers)

#     def forward(self, x):
#         x = self.stem(x)
#         x = self.layer1(x)
#         x = self.layer2(x)
#         x = self.layer3(x)
#         x = self.pool(x).flatten(1)
#         return self.fc(x)


### Setup

In [ ]:
# Model
model = LeNetModel()

# Device
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
model.to(device)
print(device)

# Loss Function
loss_fn = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters())

cuda


### Evaluation Function

In [ ]:
def evaluate(dataloader, model, loss_fn):
    model.eval()
    losses = []
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in dataloader:
            inputs, labels = batch[0].to(device), batch[1].to(device)
            preds = model(inputs)

            loss = loss_fn(preds, labels)
            losses.append(loss.item())

            # preds.shape = (32, 10) -> dim=1 : 각 10개의 클래스에 대한 확률(logit)
            # argmax : 10개의 클래스 중 가장 높은 값을 가진 클래스의 인덱스를 반환.
            pred_labels = torch.argmax(preds, dim=1)

            correct += (pred_labels == labels).sum().item()
            total += len(preds)

    avg_loss = sum(losses) / len(losses)
    acc = correct / total

    return avg_loss, acc

### Training & Validation

In [ ]:
epochs = 10
step = 0
# loss 값이 최소일 때의 상태를 저장하는 변수.
min_loss = float('inf')

for epoch in range(epochs):

    ''' Training '''
    model.train()
    for train_batch in train_dataloader:
        inputs, labels = train_batch[0].to(device), train_batch[1].to(device)
        preds = model(inputs)
        loss = loss_fn(preds, labels)

        # update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 100번 단위로 학습 진행 상태를 출력.
        step += 1
        if step % 100 == 0:
            print(f"(step {step}) train loss: {loss.item():.4f}")


    ''' Validation '''
    val_loss, val_acc = evaluate(val_dataloader, model, loss_fn)
    # 1 epoch이 완료될 때마다 loss와 accuracy 결과 출력.
    print(f"[epoch {epoch+1}/{epochs}] val loss: {val_loss:.4f}, val acc: {val_acc:.4f}\n")

    # loss가 최소일 때 상태를 저장.
    if min_loss >= val_loss:
        min_loss = val_loss
        torch.save(model.state_dict(), 'model.pt')

print("Training Finished")

(step 100) train loss: 1.0046
(step 200) train loss: 0.6153
(step 300) train loss: 0.5795
(step 400) train loss: 0.6601
[epoch 1/10] val loss: 0.5374, val acc: 0.8025

(step 500) train loss: 0.5757
(step 600) train loss: 0.6790
(step 700) train loss: 0.4588
(step 800) train loss: 0.5531
[epoch 2/10] val loss: 0.4622, val acc: 0.8280

(step 900) train loss: 0.5901
(step 1000) train loss: 0.4649
(step 1100) train loss: 0.4597
(step 1200) train loss: 0.4598
[epoch 3/10] val loss: 0.4210, val acc: 0.8480

(step 1300) train loss: 0.3756
(step 1400) train loss: 0.5422
(step 1500) train loss: 0.4028
(step 1600) train loss: 0.5081
[epoch 4/10] val loss: 0.3879, val acc: 0.8545

(step 1700) train loss: 0.5321
(step 1800) train loss: 0.4439
(step 1900) train loss: 0.3858
(step 2000) train loss: 0.4823
(step 2100) train loss: 0.4985
[epoch 5/10] val loss: 0.3633, val acc: 0.8652

(step 2200) train loss: 0.3410
(step 2300) train loss: 0.3650
(step 2400) train loss: 0.4092
(step 2500) train loss: 0

### Test

In [ ]:
# 최고 성능의 모델 가져오기
best_model = LeNetModel().to(device)
best_model.load_state_dict(torch.load('model.pt'))
test_loss, test_acc = evaluate(test_dataloader, best_model, loss_fn)
test_acc

0.8852

### Export

In [ ]:
best_model = best_model.to('cpu').eval()
example_input = (torch.randn(1, 1, 28, 28),)
exported = torch.export.export(best_model, example_input)
torch.export.save(exported, 'model.pt2')
